# Training for audio features

In [1]:
from utils import import_data_science, load_music_dataset
from utils import data_transformation
from utils import artifacts_gen
import utils


import_data_science(globals())

In [2]:
raw = np.load('/mnt/g/artifacts/signals.npy', allow_pickle=True)

signals = raw[:,3]
signals.shape


one_hot_encoding = np.vectorize(data_transformation.one_hot_function)


labels = raw[:,1]

labels = one_hot_encoding(labels)
labels.shape

(5000,)

In [3]:
from IPython.display import clear_output

X = []
Y = []

for index, (signal, label) in enumerate(zip(signals, labels)):
    try:
        # print(f"{index / labels.shape[0] * 100}%")
        signal.shape[0] == 308700
        X.append(signal)
        Y.append(label)
        # result.append((features, label))
    except Exception as e:
        # print(f"couldn't extract features, {e}")
        pass
    # clear_output(wait=True)


print(len(X), len(Y))

X_train = []
y_train = []


for (signal, label) in zip(X, Y):
    try:
        # print(f"{index / labels.shape[0] * 100}%")
        for i in range(5,10,1):
            X_train.append(signal[i*22050:(i+1)*22050])
            y_train.append(label)
        # result.append((features, label))
    except Exception as e:
        print(f"couldn't extract features, {e}")
    # clear_output(wait=True)

print(len(X_train), len(y_train))

del X
del Y
del labels
del signals
del raw


4828 4828
24140 24140


In [4]:
frame_size = 4096 
hop_size = 512
fft_window = 256
signal_duration = 14

path = os.getcwd()

In [49]:
# from IPython.display import clear_output

# X = []
# y = []

# for index, (signal, label) in enumerate(zip(signals, labels)):
#     try:
#         print(f"{index / labels.shape[0] * 100}%")
#         features = data_transformation.signal_to_features(signal)
#         X.append(features)
#         y.append(label)
#         # result.append((features, label))
#     except Exception as e:
#         print(f"couldn't extract features, {e}")
#     clear_output(wait=True)

# print("feature transformation ready")


# # Cleans out the np.nan
# # Perform only once

# print("cleaning transformation starting")

# wrong_transformation = []

# for index, sample in enumerate(X):
#     try: 
#         x = sample.shape
#         pass
#     except Exception as e:
#         wrong_transformation.append(index)

# print(f"impure signals count: {len(wrong_transformation)}")


# good_x = []
# good_y = []
# for i, v in enumerate(X):
#     if i not in wrong_transformation: 
#         good_x.append(v)

# for i, v in enumerate(y):
#     if i not in wrong_transformation:
#         good_y.append(v)

# del X
# del y
# del signals
# del raw

# print(f"ready_x: {len(good_x)}")
# print(f"ready_y: {len(good_y)}")



In [8]:
def normalize_and_make_numpy(x, y):
    x = np.array(x).astype(np.float32)
    y = np.array(y).astype(np.float32)

    return x, y

x, y = normalize_and_make_numpy(X_train,y_train)

In [9]:
x.shape, y.shape

((24140, 22050), (24140,))

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader, TensorDataset, Subset
import torch.nn.functional as F

In [11]:
class AudioDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)  # shape: (N, 1, 22050)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

from sklearn.model_selection import train_test_split

# x: shape (N, 3, 75, 20, 10), y: shape (N,)
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.01, stratify=y, random_state=42
)


train_dataset = AudioDataset(x_train, y_train)
test_dataset = AudioDataset(x_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [12]:
len(train_dataset), len(test_dataset)

(23898, 242)

In [30]:
# 0.9 0.1 split 
# 45000 -> 512 -> 5. 55%
# 45000 -> 1024 -> 512 -> 5. 58%
# 45000 -> 1024 -> 512 -> 64 -> 5. 55.8%

In [22]:
# class MusicModel(nn.Module):
#     def __init__(self):
#         super(MusicModel, self).__init__()
#         self.linear_relu_stack = nn.Sequential(
#         nn.Linear(45000, 2048),
#         nn.ReLU(),
#         nn.Linear(2048, 512),
#         nn.ReLU(),
#         nn.Linear(512, 5),
#         )
# # 1024 - 48% acc
#     def forward(self, x):
#         logits = self.linear_relu_stack(x)
        
#         return logits


# class MusicModel(nn.Module):
#     def __init__(self):
#         super(MusicModel, self).__init__()
#         self.conv_layers = nn.Sequential(
#             nn.Conv3d(3, 32, kernel_size=(3, 3, 3), padding=1),  # (C, T, H, W) → (16, ...)
#             nn.ReLU(),
#             nn.MaxPool3d((1, 2, 2)),  # reduce H, W
#             nn.Conv3d(32, 64, kernel_size=(3, 3, 3), padding=1),
#             nn.ReLU(),
#             nn.MaxPool3d((2, 2, 2)),  # reduce T, H, W
#         )
#         self.flatten = nn.Flatten()
#         self.fc = nn.Sequential(
#             nn.Linear(64 * 37 * 5 * 2, 256),  # adjust based on actual output shape
#             nn.ReLU(),
#             nn.Linear(256, 5)  # assuming 5 classes
#         )

#     def forward(self, x):
#         x = self.conv_layers(x)
#         x = self.flatten(x)
#         x = self.fc(x)
#         return x

import torch
import torch.nn as nn
import torch.nn.functional as F

class AudioCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 16, kernel_size=5, stride=1, padding=2)
        self.pool1 = nn.MaxPool1d(4)

        self.conv2 = nn.Conv1d(16, 32, kernel_size=5, stride=1, padding=2)
        self.pool2 = nn.AdaptiveAvgPool1d(1)  # Global average pooling

        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(32, 5)

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))  # (B, 16, ~5512)
        x = self.pool2(F.relu(self.conv2(x)))  # (B, 32, 1)
        x = x.view(x.size(0), -1)              # (B, 32)
        x = self.dropout(x)
        return self.fc(x)

In [29]:
torch.set_printoptions(precision=4, sci_mode=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AudioCNN().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
criterion = nn.CrossEntropyLoss()

# 5 -> good

for epoch in range(10):
    model.train()
    total_loss = 0

    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        optimizer.zero_grad()
        output = model(batch_x)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}: Loss = {total_loss / len(train_loader):.4f}")

    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = model(batch_x)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == batch_y).sum().item()
            total += batch_y.size(0)
    
    print(f"Validation Accuracy: {correct / total:.2%}")

Epoch 1: Loss = 1.4202
Validation Accuracy: 46.28%
Epoch 2: Loss = 1.3457
Validation Accuracy: 52.07%
Epoch 3: Loss = 1.3322
Validation Accuracy: 46.69%
Epoch 4: Loss = 1.3200
Validation Accuracy: 47.11%
Epoch 5: Loss = 1.3148
Validation Accuracy: 47.93%
Epoch 6: Loss = 1.3118
Validation Accuracy: 52.48%
Epoch 7: Loss = 1.3026
Validation Accuracy: 50.83%
Epoch 8: Loss = 1.2920
Validation Accuracy: 54.55%
Epoch 9: Loss = 1.2868
Validation Accuracy: 48.76%
Epoch 10: Loss = 1.2780
Validation Accuracy: 52.89%


In [57]:
torch.save(model.state_dict(), '/mnt/g/models/0.pth')


In [17]:

upload_ids = ["001","002","003"]
upload_dir = "/mnt/g/uploads/"
models_dir = "/mnt/g/models/"
artifacts_dir = "/mnt/g/song_artifacts/"
model_filename = "0.pth"
# TODO check and if not present make folders
# songslice
# videos
# power
# mels
# chroma
# mfcc


frame_size = 4096
hop_size = 512
fft_window = 512
signal_duration = 14

In [18]:
# Execution
filename = utils.find_file_by_upload_id(upload_id, upload_dir)
print(filename)

NameError: name 'upload_id' is not defined

In [101]:
import librosa as li
import utils

# Execution
signal, sr = utils.extract_signal_for_inference(filename)



In [19]:

# # Storing transformed signal
# song_location = utils.generate_audio_from_frames(f"generated_track_{upload_id}", signal, sr, artifacts_dir)

# # Frames indexes for generation of data
# frames = utils.split_signal_to_frame_indexes(frame_size, hop_size, signal)

# def extract_mels_and_generate_artifacts():
#     mels = utils.extract_mels(frames, signal)
#     utils.generate_mel_spec_images(upload_id, mels, artifacts_dir)
#     utils.generate_video_from_mel_spec_images(upload_id, frame_size, song_location, len(frames), artifacts_dir)
#     return mels

# def extract_power_spectrograms_and_generate_artifacts():
#     power_spectrograms = utils.extract_power_spectrograms(frames, signal)
#     utils.generate_power_spectr_spec_images(upload_id, power_spectrograms, artifacts_dir)
#     utils.generate_video_from_power_spetr_images(upload_id, frame_size, song_location, len(frames), artifacts_dir)
#     return power_spectrograms

# def extract_mfcc_and_generate_artifacts():
#     mfcc = utils.extract_mfccs(frames, signal)
#     utils.generate_mfcc_images(upload_id, mfcc, artifacts_dir)
#     utils.generate_video_from_mfcc_images(upload_id, frame_size, song_location, len(frames), artifacts_dir)
#     return mfcc

# mels = extract_mels_and_generate_artifacts()
# power_spectrograms = extract_power_spectrograms_and_generate_artifacts()
# mfcc = extract_mfcc_and_generate_artifacts()

In [89]:
# stacked = utils.stack_mfcc_power_spectro_mel_spectro(mfcc, mels, power_spectrograms)
# stacked.shape

# x = np.array(stacked).astype(np.float32)

# flattened = torch.from_numpy(x).reshape(-1)

# output = model(flattened)
# percentages = F.softmax(output, dim=0) * 100

NameError: name 'mfcc' is not defined

In [20]:
signal

# for i in range(5,10,1):
#             X_train.append(signal[i*22050:(i+1)*22050])

array([0.0000000e+00, 7.2819413e-09, 2.7752554e-08, ..., 4.7088382e-09,
       1.2593491e-09, 0.0000000e+00], shape=(308700,), dtype=float32)

In [113]:
import IPython.display as ipd



frames = []
for index in range(1, 14, 1):
    frames.append(signal[index*22050:(index+1)*22050])
    # ipd.Audio(data= signal[index*22050:(index+1)*22050], rate=22050)

ipd.Audio(data=frames[1], rate=22050)
    

In [118]:
ipd.Audio(data=test_dataset.X[0][0], rate=22050)
    


In [31]:
model.eval()

for upload_id in upload_ids:
    print(f"infering from {upload_id}") 
    
    filename = utils.find_file_by_upload_id(upload_id, upload_dir)
    model.eval()
    # Execution
    signal, sr = utils.extract_signal_for_inference(filename)

    
    X = []
    
    for i in range(1,14,1):
        X.append(signal[i*22050:(i+1)*22050])
    

    print(X[0].shape)

    for frame in X:
        waveform = torch.tensor(frame, dtype=torch.float32).unsqueeze(0).unsqueeze(0)  # shape (1, 1, 22050)
        print(waveform.size())
        waveform = waveform.to(device)
        with torch.no_grad():
            output = model(waveform)
            prediction = F.softmax(output, dim=1)
            # prediction = torch.argmax(output, dim=1).item()
            print(f"Predictions: {prediction * 100}")
            class_id = torch.argmax(output)
            print(utils.reverse_one_hot(class_id.item()))
# Metallica : Predicted class: tensor([[ 9.3146,  8.1708, 22.2267, 36.9664, 23.3215]], device='cuda:0')

# Predicted class: tensor([[22.6379, 16.7982, 19.2500, 18.4610, 22.8529]], device='cuda:0') # 5 epochs metallica

infering from 001
Upload couldn't be found
Upload couldn't be found
(22050,)
torch.Size([1, 1, 22050])
Predictions: tensor([[   10.7204,    38.1672,    23.5215,    27.5905,     0.0004]],
       device='cuda:0')
Hip-Hop
torch.Size([1, 1, 22050])
Predictions: tensor([[38.9952, 15.8884,  9.0800, 30.3580,  5.6784]], device='cuda:0')
Rock
torch.Size([1, 1, 22050])
Predictions: tensor([[33.6595, 18.8970, 13.6265, 30.4294,  3.3875]], device='cuda:0')
Rock
torch.Size([1, 1, 22050])
Predictions: tensor([[19.8433, 28.6234, 24.6431, 26.7895,  0.1006]], device='cuda:0')
Hip-Hop
torch.Size([1, 1, 22050])
Predictions: tensor([[28.5836, 22.9070, 19.0801, 28.3509,  1.0784]], device='cuda:0')
Rock
torch.Size([1, 1, 22050])
Predictions: tensor([[31.7039, 20.2393, 15.2191, 30.4771,  2.3605]], device='cuda:0')
Rock
torch.Size([1, 1, 22050])
Predictions: tensor([[36.3016, 16.9452, 11.4831, 29.9649,  5.3053]], device='cuda:0')
Rock
torch.Size([1, 1, 22050])
Predictions: tensor([[27.8902, 22.3741, 20.3057, 2